# 1a.01 — Reproducing RemoteCLIP Zero-Shot Results

**Goal:** Reproduce the ViT-B-32 zero-shot accuracies on RESISC45 reported in RemoteCLIP (Table III): CLIP = 60.92%, RemoteCLIP = 70.33%.

**Protocol (from the paper):** single template `"a satellite photo of {class name}."`, L2-normalized embeddings, cosine similarity, argmax prediction, default open_clip preprocessing (224×224).

**Known difference:** The paper does not state which RESISC45 images it evaluated on. We use the `timm/resisc45` test split (6,300 images), so a gap of ±1–2 points is acceptable.

## 1. Setup

### 1.1 Loading models and data
We add the repo root to the Python path so we can import our shared code from `methods/vlm.py`. Then we load both models and the RESISC45 test split.

In [1]:
import sys
sys.path.append("..")

from datasets import load_dataset
from methods.vlm import DEVICE, load_clip, load_remoteclip

clip_model, clip_preprocess, clip_tokenizer = load_clip()
rclip_model, rclip_preprocess, rclip_tokenizer = load_remoteclip(cache_dir="../cache/checkpoints")

resisc = load_dataset("timm/resisc45", split="test", cache_dir="../cache/datasets")
class_names = [name.replace("_", " ") for name in resisc.features["label"].names]

print("Device:", DEVICE)
print(len(resisc), "test images,", len(class_names), "classes")
print("First 5 classes:", class_names[:5])

/Users/yagizcetin/Documents/Github/Project-400/proj400/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No pretrained weights loaded for model 'ViT-B-32'. Model initialized randomly.


Device: mps
6300 test images, 45 classes
First 5 classes: ['airplane', 'airport', 'baseball diamond', 'basketball court', 'beach']


### 1.2 Building a stratified split
The `timm/resisc45` test split is not stratified (115–169 images per class), and neither paper states its own split. We therefore merge all 31,500 images (700 per class) and split each class 60/20/20 with a fixed seed: 420 train, 140 val, 140 test per class. The assignment is saved to `data/splits/resisc45_seed0.csv` so every later experiment uses the same split.

In [3]:
import os
import numpy as np
import pandas as pd
from datasets import concatenate_datasets
from sklearn.model_selection import train_test_split

all_parts = load_dataset("timm/resisc45", cache_dir="../cache/datasets")
full = concatenate_datasets(list(all_parts.values()))
labels = np.array(full["label"])
idx = np.arange(len(full))

train_idx, rest_idx = train_test_split(idx, test_size=0.4, stratify=labels, random_state=0)
val_idx, test_idx = train_test_split(rest_idx, test_size=0.5, stratify=labels[rest_idx], random_state=0)

split = np.empty(len(full), dtype=object)
split[train_idx], split[val_idx], split[test_idx] = "train", "val", "test"

os.makedirs("../data/splits", exist_ok=True)
pd.DataFrame({"label": labels, "split": split}).to_csv("../data/splits/resisc45_seed0.csv", index_label="index")

counts = pd.crosstab(labels, split)
print(len(full), "images in total")
print(counts.agg(["min", "max"]))

31500 images in total
col_0  test  train  val
min     140    420  140
max     140    420  140


## 2. Encoding

### 2.1 Image embeddings (cached)
We encode the 6,300 images of our stratified test split with both models. Encoding is the slowest step, so each result is saved to `cache/embeddings/`. On later runs the saved file is loaded instead of encoding again. As a sanity check, every embedding should have norm 1 because `encode_images` L2-normalizes them.

In [4]:
import torch
from methods.vlm import encode_images

test_set = full.select(test_idx)
test_labels = torch.tensor(labels[test_idx])
images = test_set["image"]

os.makedirs("../cache/embeddings", exist_ok=True)

def cached_image_embeddings(name, model, preprocess):
    path = f"../cache/embeddings/resisc45_seed0_test_{name}.pt"
    if os.path.exists(path):
        return torch.load(path)
    emb = encode_images(model, preprocess, images)
    torch.save(emb, path)
    return emb

clip_img = cached_image_embeddings("clip", clip_model, clip_preprocess)
rclip_img = cached_image_embeddings("remoteclip", rclip_model, rclip_preprocess)

print("CLIP:", clip_img.shape, "norm range:", clip_img.norm(dim=-1).min().item(), clip_img.norm(dim=-1).max().item())
print("RemoteCLIP:", rclip_img.shape, "norm range:", rclip_img.norm(dim=-1).min().item(), rclip_img.norm(dim=-1).max().item())

CLIP: torch.Size([6300, 512]) norm range: 0.9999997615814209 1.000000238418579
RemoteCLIP: torch.Size([6300, 512]) norm range: 0.9999997615814209 1.000000238418579


## 3. Zero-shot evaluation

### 3.1 Accuracy vs. paper
For each model we encode the 45 prompts `"a satellite photo of {class}."`, compute the cosine similarity between every test image and every prompt, and predict the class with the highest similarity. The ×100 logit scale and softmax from notebook 01 are not needed here because they do not change which class is the largest.

Since our test split is balanced (140 images per class), accuracy and macro accuracy are identical, so we report accuracy only. We also keep the per-class accuracies for the next plot.

In [6]:
from methods.vlm import encode_texts

def evaluate(name, img_emb, text_emb, paper_acc):
    preds = (img_emb @ text_emb.T).argmax(dim=-1)
    correct = (preds == test_labels).float()
    acc = correct.mean().item() * 100
    per_class = torch.stack([correct[test_labels == c].mean() for c in range(len(class_names))]) * 100
    print(f"{name}: ours {acc:.2f} | paper {paper_acc} | difference {acc - paper_acc:+.2f}")
    return preds, per_class

clip_text = encode_texts(clip_model, clip_tokenizer, class_names)
rclip_text = encode_texts(rclip_model, rclip_tokenizer, class_names)

clip_preds, clip_per_class = evaluate("CLIP", clip_img, clip_text, 60.92)
rclip_preds, rclip_per_class = evaluate("RemoteCLIP", rclip_img, rclip_text, 70.33)

CLIP: ours 52.65 | paper 60.92 | difference -8.27
RemoteCLIP: ours 68.54 | paper 70.33 | difference -1.79


**Result:** RemoteCLIP is within about 2 points of the paper (68.54 vs. 70.33), but CLIP is 8.3 points lower (52.65 vs. 60.92). This gap is too large to be caused by the split alone.

### 3.2 Testing the prompt hypothesis
The CLIP paper reports 60.3% on RESISC45 using an ensemble of 18 templates, very close to the 60.92% that RemoteCLIP reports for CLIP. Our hypothesis: the CLIP baseline in RemoteCLIP was evaluated with this 18-template ensemble, not the single template. To test it, we encode each class with all 18 templates (taken from the CLIP repository), average the 18 text embeddings, re-normalize, and evaluate both models again.

In [ ]:
CLIP_TEMPLATES = [
    "satellite imagery of {}.", "aerial imagery of {}.", "satellite photo of {}.",
    "aerial photo of {}.", "satellite view of {}.", "aerial view of {}.",
    "satellite imagery of a {}.", "aerial imagery of a {}.", "satellite photo of a {}.",
    "aerial photo of a {}.", "satellite view of a {}.", "aerial view of a {}.",
    "satellite imagery of the {}.", "aerial imagery of the {}.", "satellite photo of the {}.",
    "aerial photo of the {}.", "satellite view of the {}.", "aerial view of the {}.",
]

def encode_ensemble(model, tokenizer, templates):
    emb = torch.stack([encode_texts(model, tokenizer, class_names, t) for t in templates]).mean(dim=0)
    return emb / emb.norm(dim=-1, keepdim=True)

clip_text_ens = encode_ensemble(clip_model, clip_tokenizer, CLIP_TEMPLATES)
rclip_text_ens = encode_ensemble(rclip_model, rclip_tokenizer, CLIP_TEMPLATES)

clip_preds_ens, clip_pc_ens = evaluate("CLIP (18 templates)", clip_img, clip_text_ens, 60.92)
rclip_preds_ens, rclip_pc_ens = evaluate("RemoteCLIP (18 templates)", rclip_img, rclip_text_ens, 70.33)

CLIP (18 templates): ours 59.32 | paper 60.92 | difference -1.60
RemoteCLIP (18 templates): ours 69.37 | paper 70.33 | difference -0.96
